In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RAW_NUMERIC = ["distance_km", "trip_duration_hours", "payload_kg",
               "ambient_temperature_c", "idle_minutes"]
numeric_cols = RAW_NUMERIC
numeric_features = numeric_cols + ["start_hour", "average_speed_kmph"]
category_cols = ["depot_code", "vehicle_type", "route_type"]
features = numeric_features + category_cols
TARGET = "fuel_consumed_litres"

In [ ]:
def load_data(filename:str) -> pd.DataFrame:
    df=pd.read_csv(filename,dtype={ x : str for x in category_cols })
    df['trip_id']=df['trip_id'].astype('string').str.strip().replace('',pd.NA)
    df= df.drop_duplicates(subset=['trip_id'],keep='first')
    data=df['trip_start_ts'].astype('string').str.strip().replace('',pd.NA)
    data = pd.to_datetime(data,errors='coerce',format='%Y-%m-%m')
    missing = data.isna()
    data.loc[missing] =pd.to_datetime(data,errors='coerce',format='mixed',dayfirst=True)
    df['trip_start_ts']=data
    df=df.dropna(subset=['trip_start_ts'])
    for c in RAW_NUMERIC:
        df[c]=pd.to_numeric(df[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for col in category_cols:
        df[col] = (
            df[col].astype("string").str.strip().str.lower()
            .replace("", pd.NA).fillna("unknown").astype(str)
        )
    df['start_hour']=df['trip_start_ts'].dt.hour
    positive_duration = df["trip_duration_hours"].where(
        df["trip_duration_hours"] > 0
    )
    df["average_speed_kmph"] = (
        df["distance_km"] / positive_duration
    ).replace([np.inf, -np.inf], np.nan)

    return df.reset_index(drop=True)




In [ ]:
def build_pipeline(algorithm) -> Pipeline:
    preprocessor = ColumnTransformer([
        ('numeric',Pipeline([
            ('imputer',SimpleImputer(strategy='median')),
            ('scaler',StandardScaler())
        ]),numeric_features),
        ('onehot',OneHotEncoder(handle_unknown='ignore'),category_cols)
    ])
    return Pipeline([
        ('preprocessor',preprocessor),
        ('model',algorithm)
    ])

In [ ]:
def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Return (X_train, X_test, y_train, y_test).

    Return types, in order: DataFrame, DataFrame, Series, Series.
    Targets with missing, invalid or infinite values are removed.
    """
    train = load_data(train_file)
    test = load_data(test_file)
    for df in (train,test):
        df[TARGET]=pd.to_numeric(df[TARGET],errors='coerce').replace([np.inf,-np.inf],np.nan)
        df=df.dropna(subset=[TARGET])
    return (
        train[features],test[features],train[TARGET],test[TARGET]
    )

In [ ]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    models = {
        "linear_regression": LinearRegression(),
        "random_forest": RandomForestRegressor(
            n_estimators=150,
            max_depth=10,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    }
    trained_models={}
    for name,model in models.items():
        pipeline = build_pipeline(model)
        pipeline.fit(X_train,y_train)
        trained_models[name]=pipeline
    return trained_models


In [ ]:
def evaluate_models(trained_models: dict[str, Pipeline],
                    X_test: pd.DataFrame,
                    y_test: pd.Series) -> pd.DataFrame:

                    result =[]
                    for name,model in trained_models.items():
                        pred = model.predict(X_test)
                        result.append({
                            'model_name':name,
                            'mae':mean_absolute_error(y_test,pred),
                            'rmse':np.sqrt(mean_squared_error(y_test,pred)),
                            'r2':r2_score(y_test,pred)
                        })
                    df=pd.DataFrame(result).sort_values(['rmse'],ascending=[True]).reset_index(drop=True)

                    return df



In [10]:
def select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    best_model = comparison.iloc[0]['model_name']
    return (
        best_model,trained_models[best_model]
    )


In [ ]:
def predict_fuel(model: Pipeline, df: pd.DataFrame) -> pd.DataFrame:
    pred = model.predict(df)
    prediction = np.maximum(pred,0).round(1)
    return pd.DataFrame({
        "trip_id": df["trip_id"].to_numpy(),
        "predicted_fuel_consumed_litres": prediction
    })

In [12]:
def train() -> tuple[str, pd.DataFrame]:
    X_train, X_test, y_train, y_test = prepare_datasets(
        "fleet_train.csv", "fleet_test.csv"
    )
    trained_models = train_models(X_train, y_train)
    comparison = evaluate_models(trained_models, X_test, y_test)
    best_name, best_model = select_best_model(comparison, trained_models)
    joblib.dump(best_model, "best_model.pkl")
    comparison.to_csv("model_comparison.csv", index=False)
    return best_name, comparison

In [ ]:
def predict() -> pd.DataFrame:
    """Load a fitted model and return the queued prediction DataFrame."""
    df = load_data("fleet_predict.csv")
    model = joblib.load("best_model.pkl")
    result = predict_fuel(model, df)
    result.to_csv("fleet_predictions.csv", index=False)

    return result